After the following steps:
- cleaning the clapnq corpus and push it to private HF dataset
- cleaning the clapnq benchmark and push it to the private HF dataset
- chunk and embed the cleaned clapnq corpus (by sentences) to the local Qdrant vector DB

Now we can:
- Load the cleaned benchmark (train set)
- For each question in the benchmark, get the top-N docs from the vector DB and compare with the expected docs of that question from the benchmark.

In [62]:
from langchain_qdrant import QdrantVectorStore, RetrievalMode
from local_rag.core.embeddings import CustomEmbeddings
from datasets import load_dataset
from local_rag.utils.logger import get_logger
import os
from dotenv import load_dotenv
from huggingface_hub import login
from pathlib import Path
import pandas as pd
import time


qdrant_svc_url = "http://localhost:6333"
collection_name = "clapqa_corpus_cleaned"
embedding_url = "http://127.0.0.1:5002/invocations"

embeddings = CustomEmbeddings(endpoint_url=embedding_url)
vector_store = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    collection_name=collection_name,
    url=qdrant_svc_url,
    retrieval_mode=RetrievalMode.DENSE,
)


cwd = Path.cwd()
load_dotenv(cwd / "../../.env")
HF_TOKEN = os.getenv("HF_TOKEN")
login(HF_TOKEN)

logger = get_logger(__name__)
USER_NAME = "joshuale"
CLEANED_DATASET_NAME = f"{USER_NAME}/clapnq_cleaned"

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 1. Getting Data from HF

In [35]:
# Load the cleaned dataset from Hugging Face Hub
dataset_dict = load_dataset(CLEANED_DATASET_NAME)
logger.info(f"Available splits: {list(dataset_dict.keys())}")
logger.info(f"Train size: {len(dataset_dict['train'])}")
logger.info(f"Validation size: {len(dataset_dict['validation'])}")

2026-01-16 17:33:32 INFO     Available splits: ['train', 'validation']

                    INFO     Train size: 3745

                    INFO     Validation size: 600

In [52]:
# convert to a pandas dataframe for easier manipulation
train_df = pd.DataFrame(dataset_dict['train'])
def extract_answers(output_list):
    return output_list[0].get('answer', '')

def extract_selected_sentences(output_list):
    return output_list[0].get('selected_sentences', [])

train_df['answer'] = train_df['output'].apply(extract_answers)
train_df['selected_sentences'] = train_df['output'].apply(extract_selected_sentences)
train_df.head()

,id,input,passages,output,answer,selected_sentences
0,8045984229282682032,who sang love the one you're with first,"[{'title': 'Love the One You're With', 'text':...",[{'answer': '`` Love the One You're With''is a...,`` Love the One You're With''is a song by folk...,[`` Love the One You're With''is a song by fol...
1,8116539286638468011,what were the companies that built the transco...,"[{'title': 'First Transcontinental Railroad', ...",[{'answer': 'The rail line was built by three ...,The rail line was built by three private compa...,[The rail line was built by three private comp...
2,-3228156810154352180,what is the function of the human brain,"[{'title': 'Human brain', 'text': 'The human b...",[{'answer': 'The human brain is the central or...,The human brain is the central organ of the hu...,[The human brain is the central organ of the h...
3,3987125231951435424,how did france contribute to the american vict...,[{'title': 'France in the American Revolutiona...,[{'answer': 'France allied with the United Sta...,France allied with the United States during th...,[France allied with the United States during t...
4,4033741599548472305,what impact did the treaty of fontainebleau ha...,"[{'title': 'Treaty of Fontainebleau (1762)', '...",[{'answer': 'The Treaty of Fontainebleau was a...,The Treaty of Fontainebleau was a secret agree...,[The Treaty of Fontainebleau was a secret agre...


In [ ]:
# An Example (question, sentences, answer) triplet

idx = 0

sample_question = dataset_dict['train'][idx]['input']
print(sample_question)
print()
sample_sentences = dataset_dict['train'][idx]['output'][0]['selected_sentences']
print(sample_sentences)
print()
sample_answer = dataset_dict['train'][idx]['output'][0]['answer']
print(sample_answer)

who sang love the one you're with first

["`` Love the One You're With''is a song by folk rocker Stephen Stills.", "David Crosby and Graham Nash, Stills'fellow members of Crosby, Stills & Nash, provide background vocals on the song.", 'The song was also covered by a number of artists, including The Isley Brothers, Bucks Fizz, and Luther Vandross.']

`` Love the One You're With''is a song by folk rocker Stephen Stills. David Crosby and Graham Nash, Stills'fellow members of Crosby, Stills & Nash, provide background vocals on the song. The song was also covered by a number of artists, including The Isley Brothers, Bucks Fizz, and Luther Vandross.


## 2. Simple Similarity Search

In [33]:
results = vector_store.similarity_search(
    sample_question, k=3
)
print(sample_question)
# print([res.page_content for res in results])
for res in results:
    print(f"* {res.page_content} \n")
print(sample_sentences)

who sang love the one you're with first
* `` Love the One You're With''was included on the group's fifth studio album Writing on the Wall released at the end of the year. 

* 56 `` Love the One You're With''Single by Chantoozies from the album Gild the Lily Released February 1991 Format CD Single, 7 ``, 12''Recorded Metropolis Audio and Gotham Audio Genre Pop music Label Mushroom Records Songwriter (s) Stephen Stills Producer (s) Ross Inglis, Doug Brady Chantoozies singles chronology `` Walk On''(1990) `` Love the One You're With''(1991) `` I'll Be There''(1991) `` Walk On''(1990) `` Love the One You're With''(1995) `` I'll Be There''(1991) Australian group, Chantoozies released a version in February 1991. 

* `` Love the One You're With''Single by Stephen Stills from the album Stephen Stills B-side `` To a Flame''Released November 1970 Format 7-inch single Genre Folk rock Length 3: 03 Label Atlantic Songwriter (s) Stephen Stills Producer (s) Stephen Stills Bill Halverson Stephen Still

## 3. Retrieval @K=5

In [53]:
def retrieve_relevant_sentences(question:str, top_k:int=3):
    results = vector_store.similarity_search(
        question, k=top_k
    )
    return [res.page_content for res in results]

In [63]:
# perform batch retrieval on the training set
# this will take some time to complete (invoking the backend embedding service + querying Qdrant for each example)
logger.info("Starting batch retrieval on training set...")
logger.info(f"Total examples to process: {len(train_df)}")
start_time = time.time()
retrieval_results = []
for i in range(len(train_df)):
    question = train_df.iloc[i]['input']
    retrieved_sentences = retrieve_relevant_sentences(question, top_k=5)
    retrieval_results.append(retrieved_sentences)
end_time = time.time()
elapsed_time = end_time - start_time
logger.info(f"Completed batch retrieval in {elapsed_time:.2f} seconds.")

2026-01-16 17:58:35 INFO     Starting batch retrieval on training set...

                    INFO     Total examples to process: 3745

2026-01-16 18:00:08 INFO     Completed batch retrieval in 93.00 seconds.

In [64]:
train_df["retrieved_sentences_k5"] = retrieval_results

In [71]:
train_df.head()

,id,input,passages,output,answer,selected_sentences,retrieved_sentences_k5
0,8045984229282682032,who sang love the one you're with first,"[{'title': 'Love the One You're With', 'text':...",[{'answer': '`` Love the One You're With''is a...,`` Love the One You're With''is a song by folk...,[`` Love the One You're With''is a song by fol...,[`` Love the One You're With''was included on ...
1,8116539286638468011,what were the companies that built the transco...,"[{'title': 'First Transcontinental Railroad', ...",[{'answer': 'The rail line was built by three ...,The rail line was built by three private compa...,[The rail line was built by three private comp...,[Although the transcontinental railroads domin...
2,-3228156810154352180,what is the function of the human brain,"[{'title': 'Human brain', 'text': 'The human b...",[{'answer': 'The human brain is the central or...,The human brain is the central organ of the hu...,[The human brain is the central organ of the h...,[Information about the structure and function ...
3,3987125231951435424,how did france contribute to the american vict...,[{'title': 'France in the American Revolutiona...,[{'answer': 'France allied with the United Sta...,France allied with the United States during th...,[France allied with the United States during t...,"[However, after the Battles of Saratoga were c..."
4,4033741599548472305,what impact did the treaty of fontainebleau ha...,"[{'title': 'Treaty of Fontainebleau (1762)', '...",[{'answer': 'The Treaty of Fontainebleau was a...,The Treaty of Fontainebleau was a secret agree...,[The Treaty of Fontainebleau was a secret agre...,[The Treaty of Fontainebleau was a secret agre...


## 4. Retrieval Eval - Basic Metrics

### i. Hit Rate @K = 5

In [ ]:
def compute_hit_rate(
    retrieved_sentences: list[str], ground_truth_sentences: list[str]
) -> bool:
    if any(sent in retrieved_sentences for sent in ground_truth_sentences):
        return True
    else:
        return False

In [ ]:
hit_rates = []
for i in range(len(train_df)):
    retrieved = train_df.iloc[i]['retrieved_sentences_k5']
    ground_truth = train_df.iloc[i]['selected_sentences']
    hr = compute_hit_rate(retrieved, ground_truth)
    hit_rates.append(hr)

### ii. Precision @K = 5

In [72]:
import numpy as np


def compute_precision(
        retrieved_sentences: list[str], ground_truth_sentences: list[str]
) -> float:
    if not ground_truth_sentences:
        return np.nan # early return if no ground truth
    if not retrieved_sentences:
        return 0.0
    relevant_retrieved = sum(1 for sent in retrieved_sentences if sent in ground_truth_sentences)
    precision = relevant_retrieved / len(retrieved_sentences)
    return precision

In [73]:
precisions = []
for i in range(len(train_df)):
    retrieved = train_df.iloc[i]['retrieved_sentences_k5']
    ground_truth = train_df.iloc[i]['selected_sentences']
    prec = compute_precision(retrieved, ground_truth)
    precisions.append(prec)

In [79]:
# remove nan:
valid_precisions = [p for p in precisions if not np.isnan(p)]

average_precision = np.mean(valid_precisions)
print(f"Average Precision@5: {average_precision:.4f}")

Average Precision@5: 0.0874


### iii. Recall @K = 5

In [80]:
def compute_recall(
        retrieved_sentences: list[str], ground_truth_sentences: list[str]
) -> float:
    if not ground_truth_sentences:
        return np.nan # early return if no ground truth
    if not retrieved_sentences:
        return 0.0
    relevant_retrieved = sum(1 for sent in retrieved_sentences if sent in ground_truth_sentences)
    recall = relevant_retrieved / len(ground_truth_sentences)
    return recall

In [81]:
recalls = []
for i in range(len(train_df)):
    retrieved = train_df.iloc[i]['retrieved_sentences_k5']
    ground_truth = train_df.iloc[i]['selected_sentences']
    rec = compute_recall(retrieved, ground_truth)
    recalls.append(rec)

In [82]:
# remove nan:
valid_recalls = [r for r in recalls if not np.isnan(r)]

average_recall = np.mean(valid_recalls)
print(f"Average Recall@5: {average_recall:.4f}")

Average Recall@5: 0.1674


### iv. F1 @K = 5

In [85]:
def compute_f1(
        retrieved_sentences: list[str], ground_truth_sentences: list[str]
) -> float:
    precision = compute_precision(retrieved_sentences, ground_truth_sentences)
    recall = compute_recall(retrieved_sentences, ground_truth_sentences)
    if np.isnan(precision) or np.isnan(recall):
        return np.nan
    if precision + recall == 0:
        return 0.0
    f1 = 2 * (precision * recall) / (precision + recall)
    return f1


In [86]:
f1s = []
for i in range(len(train_df)):
    retrieved = train_df.iloc[i]['retrieved_sentences_k5']
    ground_truth = train_df.iloc[i]['selected_sentences']
    f1 = compute_f1(retrieved, ground_truth)
    f1s.append(f1)

# remove nan:
valid_f1s = [f for f in f1s if not np.isnan(f)]
average_f1 = np.mean(valid_f1s)
print(f"Average F1@5: {average_f1:.4f}")

Average F1@5: 0.1115


## 5. LLM-as-a-judge

### Evaluate overall context quality
- [SKIPPED] as this is not rule-based metrics but LLM-as-judge
- We first assess whether the retrieved context provides sufficient information to answer the question and view results as a pandas dataframe.


In [50]:
# context_based_evals = Dataset.from_pandas(
#     train_df,
#     data_definition=DataDefinition(text_columns=["input", "selected_sentences", "answer"]),
#     descriptors=[ContextQualityLLMEval("selected_sentences", question="input")]
# )
# context_based_evals.as_dataframe()